In [20]:
import warnings

warnings.filterwarnings("ignore")

In [21]:
import os
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [22]:
texts = [
    "Artificial Intelligence is transforming healthcare.",
    "Machine learning models learn patterns from data.",
    "Natural language processing helps computers understand text.",
    "Deep learning uses neural networks with many layers.",
    "Transformers have revolutionized modern NLP."
]

dataset = Dataset.from_dict(
    {
        "text": texts
    }
)

dataset

Dataset({
    features: ['text'],
    num_rows: 5
})

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using Device: {device}")

Using Device: cpu


In [24]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [25]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [26]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_dataset

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5
})

In [27]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [28]:
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME
)

model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [29]:
training_args = TrainingArguments(
    output_dir="./mlm_model",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    save_strategy="epoch",

    logging_steps=10,

    learning_rate=5e-5,

    weight_decay=0.01,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [31]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=5, training_loss=4.439203262329102, metrics={'train_runtime': 196.7096, 'train_samples_per_second': 0.127, 'train_steps_per_second': 0.025, 'total_flos': 1645030080000.0, 'train_loss': 4.439203262329102, 'epoch': 5.0})

In [32]:
SAVE_PATH = "./trained_mlm"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Model Saved")

Model Saved


In [33]:
text = "Transformers are [MASK] for NLP."

In [34]:
inputs = tokenizer(
    text,
    return_tensors="pt"
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

In [35]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

In [36]:
mask_token_index = torch.where(
    inputs["input_ids"] == tokenizer.mask_token_id
)[1]

In [37]:
mask_logits = logits[
    0,
    mask_token_index,
    :
]

top_tokens = torch.topk(
    mask_logits,
    k=5,
    dim=1
).indices[0]

In [38]:
print("Top Predictions:\n")

for token in top_tokens:
    word = tokenizer.decode([token])

    print(word)

Top Predictions:

used
available
built
designed
made


In [39]:

best_token = top_tokens[0]

predicted_word = tokenizer.decode(
    [best_token]
)

completed_sentence = text.replace(
    tokenizer.mask_token,
    predicted_word
)

print(completed_sentence)

Transformers are used for NLP.
